---
short_title: Datenstrukturen
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Datenstrukturen: Dictionaries und `collections`

Die manuelle Analyse eines Datensatzes lebt davon, Daten sinnvoll *abzulegen*.
Eine Liste eignet sich für eine Reihenfolge von Werten, ein Dictionary dagegen
für Werte, die man über einen Namen (Schlüssel) anspricht. In diesem Kapitel
schauen wir uns Dictionaries und die wichtigsten Werkzeuge aus dem Modul
`collections` systematisch an.

:::{seealso} Vorwissen
:icon: false

Listen, Tupel und einfache Dictionaries kennen Sie aus dem Projekt
[Taschenrechner I](../../020-Projekt_Taschenrechner_I/000-Einleitung.md).
Falls Sie eine Auffrischung brauchen, sehen Sie sich dort die Einheit zu
Datenstrukturen an.
:::

## Dictionaries erzeugen

Ein Dictionary ist eine Sammlung von **Schlüssel-Wert-Paaren**. Jeder Schlüssel
kommt genau einmal vor und verweist auf einen Wert. Im Bibliothekskontext bietet
es sich an, einen Datensatz als Dictionary abzulegen: Die Spaltennamen sind die
Schlüssel, die Werte einer Zeile sind die Werte.

Dictionaries werden mit geschweiften Klammern `{ ... }` geschrieben.

In [1]:
# Ein einzelnes Buch als Dictionary: Schlüssel -> Wert
buch = {
    "isbn": "9780241951446",
    "author": "Murakami, Haruki",
    "year": 1984,
    "title": "Norwegian Wood",
}
print(buch)

{'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': 1984, 'title': 'Norwegian Wood'}


In [2]:
# Es gibt mehrere Wege, ein Dictionary zu erzeugen
leer = {}
aus_paaren = dict([("isbn", "9780241951446"), ("titel", "Norwegian Wood")])
aus_schluesselworten = dict(isbn="9780241951446", jahr=1984)

print("leer:              ", leer)
print("aus Paaren:        ", aus_paaren)
print("aus Schlüsselwörter:", aus_schluesselworten)

leer:               {}
aus Paaren:         {'isbn': '9780241951446', 'titel': 'Norwegian Wood'}
aus Schlüsselwörter: {'isbn': '9780241951446', 'jahr': 1984}


## Zugriff auf Werte

Auf einen Wert greift man über den Schlüssel in eckigen Klammern zu. Existiert
der Schlüssel nicht, wirft Python einen `KeyError`.

In [3]:
print(buch["author"])
print(buch["year"])

# Mit dem Wert kann gerechnet werden, weil er eine Zahl ist
print("Erscheinungsjahr + 1 =", buch["year"] + 1)

Murakami, Haruki
1984
Erscheinungsjahr + 1 = 1985


In [4]:
# Ein fehlender Schlüssel löst einen KeyError aus.
# Wir fangen den Fehler hier ab, damit das Notebook weiterläuft.
try:
    print(buch["verlag"])
except KeyError as fehler:
    print(f"KeyError: der Schlüssel {fehler} ist nicht vorhanden.")

KeyError: der Schlüssel 'verlag' ist nicht vorhanden.


Mit der Methode `.get()` vermeiden wir den `KeyError`. Fehlt der Schlüssel,
liefert `.get()` den Wert `None` – oder einen von uns angegebenen Standardwert.
Das ist besonders praktisch beim Zählen und Aufsummieren.

In [5]:
print(buch.get("author"))
print(buch.get("verlag"))                # None
print(buch.get("verlag", "unbekannt"))   # Standardwert

Murakami, Haruki
None
unbekannt


## Schlüssel, Werte und Paare

Dictionaries lassen sich auf drei Arten durchlaufen: über die Schlüssel
(`.keys()`), die Werte (`.values()`) oder die Paare (`.items()`). Meistens
brauchen wir `.items()`, weil wir Schlüssel und Wert gemeinsam verarbeiten.

In [6]:
print("Schlüssel:", list(buch.keys()))
print("Werte:    ", list(buch.values()))

for schluessel, wert in buch.items():
    print(f"{schluessel:>7}: {wert}")

Schlüssel: ['isbn', 'author', 'year', 'title']
Werte:     ['9780241951446', 'Murakami, Haruki', 1984, 'Norwegian Wood']
   isbn: 9780241951446
 author: Murakami, Haruki
   year: 1984
  title: Norwegian Wood


## Dictionaries verändern

Dictionaries sind **veränderbar** (`mutable`): Einträge lassen sich hinzufügen,
überschreiben und löschen. Mit `in` prüfen wir, ob ein Schlüssel vorhanden ist.

In [7]:
buch["verlag"] = "Hanser"            # neuen Eintrag hinzufügen
buch["year"] = buch["year"] + 1      # vorhandenen Eintrag überschreiben
del buch["isbn"]                     # Eintrag löschen

print("isbn vorhanden?", "isbn" in buch)
print(buch)

isbn vorhanden? False
{'author': 'Murakami, Haruki', 'year': 1985, 'title': 'Norwegian Wood', 'verlag': 'Hanser'}


## Verschachtelte Dictionaries

Ein Dictionary darf als Wert wieder ein Dictionary enthalten. So bilden wir
eine kleine Datenbank ab: Der Schlüssel ist die ISBN, der Wert ein Dictionary
mit den Feldern des Werks. Man spricht von **verschachtelten** Dictionaries.

In [8]:
bestand = {
    "9780241951446": {
        "title": "Norwegian Wood",
        "author": "Murakami, Haruki",
        "year": 1984,
        "sales": [61, 55, 48],
    },
    "9783518587249": {
        "title": "Der neue Geist des Kapitalismus",
        "author": "Boltanski, Luc",
        "year": 1999,
        "sales": [40, 33, 29],
    },
}

for isbn, daten in bestand.items():
    gesamt = sum(daten["sales"])
    print(f"{daten['title']} ({daten['author']}): {gesamt} Verkäufe")

Norwegian Wood (Murakami, Haruki): 164 Verkäufe
Der neue Geist des Kapitalismus (Boltanski, Luc): 102 Verkäufe


## Häufigkeiten zählen mit `collections.Counter`

Beim Auswerten von Daten müssen wir ständig zählen: Wie oft kommt eine
Autor\*in vor? Wie häufig ein Jahr? Das Modul `collections` liefert dafür
fertige Werkzeuge. {term}`Counter` zählt die Elemente einer Liste und bietet mit
`.most_common()` direkt eine Rangliste.

In [9]:
from collections import Counter

verkaeufe = [
    "Murakami, Haruki", "Murakami, Haruki", "Murakami, Haruki",
    "Kafka, Franz", "Kafka, Franz",
    "Boltanski, Luc",
    "Murakami, Haruki",
]

zaehler = Counter(verkaeufe)
print(zaehler)
print("Murakami:", zaehler["Murakami, Haruki"])
print("Die drei häufigsten:", zaehler.most_common(3))

Counter({'Murakami, Haruki': 4, 'Kafka, Franz': 2, 'Boltanski, Luc': 1})
Murakami: 4
Die drei häufigsten: [('Murakami, Haruki', 4), ('Kafka, Franz', 2), ('Boltanski, Luc', 1)]


## `collections.defaultdict`

Ein {term}`defaultdict` verhält sich wie ein normales Dictionary, erzeugt aber
fehlende Werte automatisch. Gibt man als Standard `list` an, entsteht zu jedem
neuen Schlüssel sofort eine leere Liste. Das erspart die `if`-Abfrage bzw. den
Umweg über `.get()` und eignet sich ideal zum Gruppieren.

In [10]:
from collections import defaultdict

verkaeufe = [
    ("Murakami, Haruki", 2020, 61),
    ("Murakami, Haruki", 2021, 55),
    ("Kafka, Franz", 2020, 12),
    ("Kafka, Franz", 2021, 19),
    ("Boltanski, Luc", 2020, 40),
]

# defaultdict(list) erzeugt fehlende Werte automatisch als leere Liste
pro_autor = defaultdict(list)
for autor, jahr, anzahl in verkaeufe:
    pro_autor[autor].append(anzahl)

for autor, zahlen in pro_autor.items():
    print(f"{autor}: {zahlen} -> Summe {sum(zahlen)}")

Murakami, Haruki: [61, 55] -> Summe 116
Kafka, Franz: [12, 19] -> Summe 31
Boltanski, Luc: [40] -> Summe 40


## `collections.namedtuple` (kurz)

Ein `namedtuple` ist ein Tupel, dessen Felder einen Namen haben. Es ist
kompakt, unveränderlich und selbst-dokumentierend – nützlich für kleine,
feste Datensätze. Für veränderliche Daten nehmen wir lieber ein Dictionary.

In [11]:
from collections import namedtuple

Buch = namedtuple("Buch", ["isbn", "author", "title"])
b = Buch("9780241951446", "Murakami, Haruki", "Norwegian Wood")

print(b.title)   # Zugriff über den Namen
print(b[1])      # Zugriff über den Index funktioniert weiterhin

Norwegian Wood
Murakami, Haruki


## Übung: Anzahl Bücher pro Autor\*in 💪

:::{exercise} Anzahl Einträge pro Autor\*in
:label: dict-autor-buecher

Erzeugen Sie aus der folgenden Liste ein Dictionary, das jeder Autor\*in die
Anzahl ihrer bzw. seiner Einträge zuordnet:

```python
autoren = [
    "Murakami, Haruki", "Kafka, Franz", "Kafka, Franz",
    "Boltanski, Luc", "Murakami, Haruki", "Murakami, Haruki",
]
```

Lösen Sie die Aufgabe **einmal von Hand** (mit `.get()`) und **einmal mit**
`collections.Counter`. Vergleichen Sie die beiden Ergebnisse.
:::

:::{solution} dict-autor-buecher
:class: dropdown

**Variante 1 – von Hand mit `.get()`:**

```python
zaehler = {}
for autor in autoren:
    zaehler[autor] = zaehler.get(autor, 0) + 1
print(zaehler)
```

**Variante 2 – mit `Counter`:**

```python
from collections import Counter
print(dict(Counter(autoren)))
```

Beide Varianten liefern dasselbe Ergebnis. `Counter` ist kürzer, die
`.get()`-Variante zeigt aber, was im Hintergrund passiert.
:::

## Zusammenfassung

- Dictionaries speichern **Schlüssel-Wert-Paare**; der Zugriff erfolgt über den
  Schlüssel.
- Ein fehlender Schlüssel führt zum `KeyError`; `.get()` liefert stattdessen
  `None` oder einen Standardwert.
- `.keys()`, `.values()` und `.items()` durchlaufen die Bestandteile.
- Verschachtelte Dictionaries bilden komplexe Datensätze ab.
- `Counter`, `defaultdict` und `namedtuple` aus `collections` lösen häufige
  Aufgaben elegant: zählen, gruppieren und strukturieren.